# Eclat / Julia — Colab demo theo kế hoạch 3 module

**Template, chưa phải demo hoàn chỉnh.** A: thuật toán/performance; B: test,
Groceries, UCI Splice DNA và Colab. M1 28/10–03/11, M2 04/11–10/11,
M3 11/11–17/11/2026. Các mốc là lịch nội bộ.

Notebook dùng kernel Python để setup/gọi Julia bằng subprocess. Các cờ mặc định
bật miner/BitVector đã triển khai và tắt phần ứng dụng/benchmark cần chạy riêng; chạy các ô setup thành công
không có nghĩa đã đạt gate đồ án. B phải thay TODO và cold-run trước khi nộp.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import urllib.request

REPO_URL = "https://github.com/phuctran33/-DM-Eclat.git"
JULIA_VERSION = "1.11.4"
RUN_MINER = True
RUN_SPMF = False
RUN_OPTIMIZED = True
RUN_APPLICATIONS = False
RUN_BENCHMARKS = False
FINAL_DELIVERY_VALIDATION = False
SEED = 42

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/eclat-project")
    if not (ROOT / "Project.toml").exists():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("Điền URL GitHub public thật của nhóm trước khi chạy Colab.")
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    if not shutil.which("julia"):
        archive = Path("/content/julia-runtime.tar.gz")
        runtime = Path("/content/julia-runtime")
        runtime.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz",
            archive,
        )
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(runtime), "--strip-components=1"], check=True)
        os.environ["PATH"] = str(runtime / "bin") + os.pathsep + os.environ["PATH"]
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Project.toml").exists()), None)
    if ROOT is None:
        raise RuntimeError("Mở notebook từ repo root hoặc notebooks/.")

os.chdir(ROOT)
JULIA = shutil.which("julia")
if not JULIA:
    raise RuntimeError("Cài Julia 1.9+ và thêm vào PATH khi chạy local.")

def run_julia(*arguments, env=None):
    return subprocess.run(
        [JULIA, "--startup-file=no", f"--project={ROOT}", *map(str, arguments)],
        cwd=ROOT, env={**os.environ, **(env or {})}, check=True,
    )

run_julia("--version")
run_julia("-e", "using Pkg; Pkg.instantiate()")
print("ROOT:", ROOT, "SEED:", SEED)

## M1 — kiểm thử hạ tầng và CLI

Các test mặc định kiểm cả parser, schema, list/bitset, oracle và năm references SPMF thật.
Expected toy tự đếm độc lập ở 40% vẫn tách biệt với references JAR-generated có manifest.

In [ ]:
run_julia("tests/runtests.jl")
run_julia("scripts/mine.jl", "--help")
print((ROOT / "data/toy/basic.txt").read_text())
print("Expected độc lập ở minsup 40%:")
print((ROOT / "data/toy/basic.expected.txt").read_text())

## M1 — Eclat basic và SPMF oracle (B kiểm, A sửa)

Sau khi cài đặt basic, bật RUN_MINER. Tải JAR chính thức vào tools/spmf.jar,
ghi version/hash và tạo đủ năm file references ở minsup 40%, rồi bật RUN_SPMF.
Không dùng expected tự đếm để giả lập reference đã chạy SPMF. Đọc
tests/reference/spmf/README.md. Gate M1 yêu cầu so khớp toàn bộ 100%.

In [ ]:
if RUN_MINER:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", "results/basic.txt",
              "--minsup", "40%", "--backend", "list")
    run_julia("scripts/compare_spmf.jl", "results/basic.txt", "data/toy/basic.expected.txt")
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1"})
else:
    print("Miner đã triển khai; RUN_MINER đang tắt trong phiên này.")

if RUN_SPMF:
    subprocess.run(["python3", "scripts/prepare_spmf.py", "--references"], check=True)
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1"})
else:
    print("Không tạo lại SPMF trong phiên này; 5 references thật và provenance đã có trong repo.")

## M2 — BitVector và dữ liệu ứng dụng

Chỉ mở sau gate correctness M1. B điền data/manifest.csv và download cells
từ snapshot công khai thật (ưu tiên HF/Kaggle), kiểm hash và schema. Code chuẩn bị
phải viết trong notebook hoặc import helpers Julia của repo.

Groceries: một basket → transaction, mapping tên ổn định; miner nhóm → rules
support/confidence/lift → top-10 theo lift và giải thích 3 luật.

DNA: UCI Splice → 12 cặp (position,base)/mẫu; nhãn EI/IE/N giữ riêng. Áp dụng
cửa sổ cố định 25:36, thống kê ambiguity và class counts. So support của pattern
2–4 item trên cả ba lớp, đếm lại candidate infrequent ở lớp khác, không gán zero.

In [ ]:
if RUN_OPTIMIZED:
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
else:
    print("RUN_OPTIMIZED đang tắt; dùng full suite để kiểm cả hai backend.")

if RUN_APPLICATIONS:
    # TODO B: replace with public, version-pinned download + hash verification.
    # These Julia helpers are the preparation entry points, currently TODO.
    run_julia("-e", 'using EclatProject; groceries_transactions("data/applications/groceries.csv")')
    run_julia("-e", 'using EclatProject; splice_transactions("data/applications/splice.data"; positions=25:36)')
    raise NotImplementedError(
        "TODO M2: nối preprocessing → mine(list/bitset) → rules/DNA tables → CSV → diễn giải; "
        "xóa exception này chỉ sau khi pipeline đầy đủ đã chạy thật."
    )
else:
    print("PENDING: Groceries và Splice DNA cần dữ liệu thật và triển khai helpers/pipeline.")

## M3 — benchmark và hình/bảng báo cáo

A: configs/experiments.toml gồm Chess, Mushroom, Retail, T10I4D100K, sáu minsup/tập.
Runner A đã triển khai: benchmark_runner.jl gọi controller Python, SPMF, warm-up/repeats, timeout và OS peak RSS.
Lưu raw CSV/log và vẽ runtime/output size; memory ở minsup trung bình; scalability
Retail 10/25/50/75/100%; synthetic width tăng dần. B đối chiếu correctness benchmark.

Không trộn timer Julia read+mine với SPMF timer khác phạm vi. Allocations không phải
peak RAM. Xem docs/experiments.md. Chèn số liệu thật vào Report.tex ba chương.

In [ ]:
if RUN_BENCHMARKS:
    # Real A pipeline; Java >=21, Python >=3.11. Full grid can take several minutes.
    subprocess.run(["python3", "scripts/prepare_spmf.py"], check=True)
    subprocess.run(["python3", "scripts/acquire_benchmarks.py"], check=True)
    run_julia("scripts/benchmark_runner.jl", "--output", "results/notebook-run")
    subprocess.run(["python3", "-m", "pip", "install", "-r", "requirements-plot.txt"], check=True)
    subprocess.run(["python3", "scripts/plot_results.py", "--results", "results/notebook-run"], check=True)
else:
    print("Benchmark pipeline đã có; bật RUN_BENCHMARKS để chạy lại vào results/notebook-run.")


## Final cold-run — B chủ trì, A review

Trước nộp: bỏ mọi TODO cần thực thi, bật đủ cờ, restart runtime và Run all.
Đặt FINAL_DELIVERY_VALIDATION=True để kiểm artifacts cơ bản; vẫn cần reviewer kiểm
chất lượng nội dung, đủ đồ thị, đúng metrics và public links. Report phải 10–15
trang IEEE tiếng Việt, paper gốc + ≥3 nguồn khác. README phải có public Colab URL.

In [ ]:
if FINAL_DELIVERY_VALIDATION:
    assert all([RUN_MINER, RUN_SPMF, RUN_OPTIMIZED, RUN_APPLICATIONS, RUN_BENCHMARKS]), "Chưa bật đủ pipeline."
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
    for relative in ["docs/Report.pdf", "results/applications/groceries_rules.csv",
                     "results/applications/dna_patterns.csv"]:
        assert (ROOT / relative).is_file(), f"Thiếu artifact: {relative}"
    print("Đã qua kiểm tra artifacts cơ bản; A/B tiếp tục rà checklist DELIVERY.md.")
else:
    print("TEMPLATE ONLY: chưa cold-run nghiệm thu đồ án, chưa có kết quả cuối.")